# 19 · Demo Clips Selection & Manifest
Select 10 representative test/out-of-domain audio clips from US8K fold 10 and ESC-50 (with duration >= 3 s, random_state=42), copy them to `demo_samples/audio/` using standardized naming `<true_class>_<source>_<original_name>.wav`, verify byte-identity against source files, run `predict_audio` end-to-end, and export `demo_samples/audio/manifest.csv`.

In [1]:
# Cell 1 – interpreter setup, imports, and input SHA256 asserts
import sys
from pathlib import Path
import hashlib
import shutil
import time

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import numpy as np
import pandas as pd
import soundfile as sf
from src.models.audio import predict_audio

print('Successfully imported predict_audio, soundfile, and dependencies')

def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_SPLIT_SHA  = '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b'

labels_sha = compute_sha256(labels_path)
split_sha = compute_sha256(split_path)

print(f'labels.csv sha256       : {labels_sha}')
print(f'split_indices.npz sha256: {split_sha}')

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert split_sha == EXPECTED_SPLIT_SHA, f'split_indices.npz SHA mismatch: {split_sha}'
print('Input SHA256 asserts passed ✓')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported predict_audio, soundfile, and dependencies
labels.csv sha256       : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
split_indices.npz sha256: 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
Input SHA256 asserts passed ✓


In [2]:
# Cell 2 – choose 10 representative clips with random_state=42 (duration >= 3 s)
labels = pd.read_csv(REPO_ROOT / 'data/processed/labels.csv')
with np.load(REPO_ROOT / 'data/processed/split_indices.npz') as sp:
    train_idx = set(sp['train_idx'])
    test_idx = set(sp['test_idx'])
    esc_idx = set(sp['esc_idx'])

f10_pool = labels.loc[sorted(test_idx)].copy()
esc_pool = labels.loc[sorted(esc_idx)].copy()

def pick_clips(df_pool, query_str, n_clips, seed=42):
    cand = df_pool.query(query_str).copy()
    # Compute duration with soundfile.info (no full audio waveform load)
    cand['duration'] = [sf.info(REPO_ROOT / p).duration for p in cand['path']]
    ge3 = cand[cand['duration'] >= 3.0]
    pool = ge3 if len(ge3) >= n_clips else cand
    return pool.sample(n=n_clips, random_state=seed)

# Fold 10 selections (US8K):
# 2 construction (one drilling, one jackhammer)
c_drill = pick_clips(f10_pool, 'original_class == "drilling"', 1, seed=42)
c_jack  = pick_clips(f10_pool, 'original_class == "jackhammer"', 1, seed=42)
# 2 hvac
c_hvac  = pick_clips(f10_pool, 'app_class == "hvac"', 2, seed=42)
# 2 music_loudspeaker
c_music = pick_clips(f10_pool, 'app_class == "music_loudspeaker"', 2, seed=42)
# 1 siren
c_siren = pick_clips(f10_pool, 'original_class == "siren"', 1, seed=42)
# 1 traffic_horn (car_horn)
c_horn  = pick_clips(f10_pool, 'original_class == "car_horn"', 1, seed=42)

# ESC-50 selections:
# 1 siren
c_esc_siren = pick_clips(esc_pool, 'original_class == "siren"', 1, seed=42)
# 1 traffic_horn (car_horn)
c_esc_horn  = pick_clips(esc_pool, 'original_class == "car_horn"', 1, seed=42)

chosen_df = pd.concat([c_drill, c_jack, c_hvac, c_music, c_siren, c_horn, c_esc_siren, c_esc_horn])
assert len(chosen_df) == 10, f'Expected 10 chosen clips, got {len(chosen_df)}'

# Assert split isolation
for idx in chosen_df[chosen_df['source'] == 'us8k'].index:
    assert idx in test_idx, f'Clip {idx} not in fold 10 test_idx!'
    assert idx not in train_idx, f'Clip {idx} found in train_idx!'

for idx in chosen_df[chosen_df['source'] == 'esc50'].index:
    assert idx in esc_idx, f'Clip {idx} not in esc_idx!'
    assert idx not in train_idx, f'Clip {idx} found in train_idx!'

print(f'Successfully selected {len(chosen_df)} clips across all required categories.')
print('\n=== Selected Demo Clips Summary ===')
summary_cols = ['source', 'fold', 'original_class', 'app_class', 'duration', 'path']
print(chosen_df[summary_cols].to_string())
print('\nCell 2 asserts passed: all clips belong to test/esc sets, zero training overlap ✓')

Successfully selected 10 clips across all required categories.

=== Selected Demo Clips Summary ===
     source  fold   original_class          app_class  duration                                                  path
6332   us8k    10         drilling       construction      4.00   data/raw/UrbanSound8K/audio/fold10/99192-4-0-40.wav
2177   us8k    10       jackhammer       construction      4.00  data/raw/UrbanSound8K/audio/fold10/162134-7-15-4.wav
5518   us8k    10  air_conditioner               hvac      4.00   data/raw/UrbanSound8K/audio/fold10/73524-0-0-92.wav
3565   us8k    10  air_conditioner               hvac      4.00   data/raw/UrbanSound8K/audio/fold10/189985-0-0-0.wav
5961   us8k    10     street_music  music_loudspeaker      4.00   data/raw/UrbanSound8K/audio/fold10/83191-9-0-11.wav
2619   us8k    10     street_music  music_loudspeaker      4.00   data/raw/UrbanSound8K/audio/fold10/171478-9-0-4.wav
1948   us8k    10            siren              siren      4.00  data/raw/

In [3]:
# Cell 3 – copy clips to demo_samples/audio/ and verify byte-identity (sha256)
demo_audio_dir = REPO_ROOT / 'demo_samples/audio'
demo_audio_dir.mkdir(parents=True, exist_ok=True)

copied_entries = []

for idx, row in chosen_df.iterrows():
    src_path = REPO_ROOT / row['path']
    orig_name = Path(row['path']).name
    dst_name = f"{row['app_class']}_{row['source']}_{orig_name}"
    dst_path = demo_audio_dir / dst_name
    
    shutil.copyfile(src_path, dst_path)
    
    # Verify byte-identical copy
    src_hash = compute_sha256(src_path)
    dst_hash = compute_sha256(dst_path)
    assert src_hash == dst_hash, f'Byte mismatch for {dst_name}: {src_hash} != {dst_hash}'
    
    copied_entries.append({
        'index': idx,
        'dst_name': dst_name,
        'dst_path': dst_path,
        'src_path': src_path,
        'sha256': dst_hash,
        'app_class': row['app_class'],
        'original_class': row['original_class'],
        'source': row['source'],
        'fold': row['fold']
    })

# Assert exactly 10 .wav files in demo_samples/audio/
wav_files = sorted(demo_audio_dir.glob('*.wav'))
print(f'Total .wav files in {demo_audio_dir.relative_to(REPO_ROOT)}: {len(wav_files)}')
assert len(wav_files) == 10, f'Expected exactly 10 .wav files, got {len(wav_files)}'

print('\n=== Copied Audio Files and Hashes ===')
for entry in copied_entries:
    rel_dst = entry['dst_path'].relative_to(REPO_ROOT)
    print(f'{str(rel_dst):55s}: {entry["sha256"]}')

print('\nCell 3 asserts passed: exactly 10 .wav files created and verified byte-identical to source ✓')

Total .wav files in demo_samples\audio: 10

=== Copied Audio Files and Hashes ===
demo_samples\audio\construction_us8k_99192-4-0-40.wav  : 1e61c260b6eff8d21767744f21b8112384abd789fa1e511a05ef3e47000b005d
demo_samples\audio\construction_us8k_162134-7-15-4.wav : 6003bbd23500fea0b31db1a54c7f507aa0e07d52332691c59170da7c51c156c7
demo_samples\audio\hvac_us8k_73524-0-0-92.wav          : 8f8ea1914672d306f4d98b075c8b9ada4a81e16f30789df0bc632b1ae2e5ba03
demo_samples\audio\hvac_us8k_189985-0-0-0.wav          : ca39039a8ac41dd58d9c46ac86995e39484fd78743ea4110a29611cd1bfa3233
demo_samples\audio\music_loudspeaker_us8k_83191-9-0-11.wav: faf7a93561c5939c035b2f5541abc79757005c2b43b44e20fef8866990908509
demo_samples\audio\music_loudspeaker_us8k_171478-9-0-4.wav: 1f4aa9f767b2e46677072b8773d58aba1683de653cd61f4e30e3051ff95168b5
demo_samples\audio\siren_us8k_159742-8-0-12.wav        : 2e854ff2798841c1fca4fa0900fd64a0bdc51e1e47e810db4a7515549aa21272
demo_samples\audio\traffic_horn_us8k_24076-1-1-0.wav   : 6

In [4]:
# Cell 4 – run predict_audio on each copy, display evaluation table, print count correct
eval_rows = []
n_correct = 0

for entry in copied_entries:
    dst_path = entry['dst_path']
    true_class = entry['app_class']
    
    res = predict_audio(dst_path)
    is_correct = (res['type'] == true_class)
    if is_correct:
        n_correct += 1
        
    eval_rows.append({
        'file_name': entry['dst_name'],
        'true_class': true_class,
        'predicted_type': res['type'],
        'confidence': res['confidence'],
        'relative_intensity': res['relative_intensity'],
        'correct': 'yes' if is_correct else 'no'
    })

df_eval = pd.DataFrame(eval_rows)
print('=== predict_audio Evaluation on Demo Clips ===')
print(df_eval.to_string(index=False, formatters={
    'confidence': '{:.4f}'.format,
    'relative_intensity': '{:.4f}'.format
}))

print(f'\nTotal Correct Predictions: {n_correct} / {len(copied_entries)} ({n_correct/len(copied_entries)*100:.1f}%)')
print('Cell 4 completed ✓')

=== predict_audio Evaluation on Demo Clips ===
                              file_name        true_class    predicted_type confidence relative_intensity correct
     construction_us8k_99192-4-0-40.wav      construction      construction     0.9885             0.6246     yes
    construction_us8k_162134-7-15-4.wav      construction      construction     0.7868             0.6836     yes
             hvac_us8k_73524-0-0-92.wav              hvac              hvac     0.6903             0.7355     yes
             hvac_us8k_189985-0-0-0.wav              hvac music_loudspeaker     0.9370             0.6015      no
music_loudspeaker_us8k_83191-9-0-11.wav music_loudspeaker             siren     0.8094             0.9045      no
music_loudspeaker_us8k_171478-9-0-4.wav music_loudspeaker music_loudspeaker     0.4426             0.1407     yes
           siren_us8k_159742-8-0-12.wav             siren             siren     1.0000             0.0000     yes
      traffic_horn_us8k_24076-1-1-0.wav  

In [5]:
# Cell 5 – generate demo_samples/audio/manifest.csv and print sha256
manifest_rows = []
for entry in copied_entries:
    manifest_rows.append({
        'file': entry['dst_name'],
        'source': entry['source'],
        'original_path': str(entry['src_path'].relative_to(REPO_ROOT)).replace('\\', '/'),
        'fold': entry['fold'],
        'true_class': entry['app_class'],
        'original_class': entry['original_class']
    })

df_manifest = pd.DataFrame(manifest_rows)
manifest_path = demo_audio_dir / 'manifest.csv'
df_manifest.to_csv(manifest_path, index=False)

assert manifest_path.exists(), f'Manifest not found at {manifest_path}'
assert len(df_manifest) == 10, f'Expected 10 rows in manifest, got {len(df_manifest)}'

print(f'Saved manifest to: {manifest_path.relative_to(REPO_ROOT)}')
print('\n=== demo_samples/audio/manifest.csv Contents ===')
print(df_manifest.to_string(index=False))

manifest_sha = compute_sha256(manifest_path)
print(f'\nmanifest.csv sha256: {manifest_sha}')
print('Cell 5 completed ✓')

Saved manifest to: demo_samples\audio\manifest.csv

=== demo_samples/audio/manifest.csv Contents ===
                                   file source                                        original_path  fold        true_class  original_class
     construction_us8k_99192-4-0-40.wav   us8k  data/raw/UrbanSound8K/audio/fold10/99192-4-0-40.wav    10      construction        drilling
    construction_us8k_162134-7-15-4.wav   us8k data/raw/UrbanSound8K/audio/fold10/162134-7-15-4.wav    10      construction      jackhammer
             hvac_us8k_73524-0-0-92.wav   us8k  data/raw/UrbanSound8K/audio/fold10/73524-0-0-92.wav    10              hvac air_conditioner
             hvac_us8k_189985-0-0-0.wav   us8k  data/raw/UrbanSound8K/audio/fold10/189985-0-0-0.wav    10              hvac air_conditioner
music_loudspeaker_us8k_83191-9-0-11.wav   us8k  data/raw/UrbanSound8K/audio/fold10/83191-9-0-11.wav    10 music_loudspeaker    street_music
music_loudspeaker_us8k_171478-9-0-4.wav   us8k  data/raw/Ur